# Шаг 5. Интерпретируемость ИИ и бизнес-аналитика (Explainable AI / XAI)

**Цель модуля:** Раскрыть принцип работы обученного алгоритма машинного обучения "черного ящика" (Black-Box). Извлечь глобальную важность признаков (какие факторы больше всего влияют на отток в целом) и построить локальные объяснения для конкретных клиентов с помощью библиотеки SHAP (SHapley Additive exPlanations), чтобы передать бизнесу конкретные рычаги управления оттоком.

---

## 📥 Шаг 1. Инициализация окружения и загрузка артефактов

Импортируем библиотеку `shap` для продвинутой интерпретации, а также восстановим из файлов финальную обученную модель и тестовую выборку.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
import warnings

warnings.filterwarnings('ignore')

PROCESSED_DIR = os.path.join(".", "data", "processed")
MODELS_DIR = os.path.join(".", "models")

# Загрузка тестовой выборки (нам нужны данные для объяснений)
try:
    X_test = pd.read_csv(os.path.join(PROCESSED_DIR, "X_test.csv"))
    print(f"✅ Тестовая выборка загружена. Размер: {X_test.shape}")
except FileNotFoundError:
    print("⚠️ Файл X_test.csv не найден. Вернитесь к ноутбуку 02_Preprocessing.")

# Загрузка финальной модели
model_path = os.path.join(MODELS_DIR, "best_churn_model.pkl")
try:
    final_model = joblib.load(model_path)
    print("✅ Финальная модель успешно десериализована и готова к анализу.")
except FileNotFoundError:
    print("⚠️ Файл best_churn_model.pkl не найден. Вернитесь к ноутбуку 04_Final_Model и выполните сохранение.")


---

## 🛠 ЗАДАНИЕ 1: Глобальная важность признаков (Feature Importance)
**Бизнес-контекст:** Первое, что спросит бизнес-заказчик: "Какие показатели сильнее всего драйвят отток?". Алгоритмы на основе деревьев (Random Forest, Gradient Boosting) имеют встроенный механизм расчета полезности каждого признака во время построения ветвей.

**Инструкция (TODO):**
1. Извлеките массив важности признаков из обученной модели (атрибут `.feature_importances_`).
2. Сопоставьте эти значения с названиями колонок датафрейма `X_test.columns`.
3. Постройте горизонтальную столбчатую диаграмму (Bar Plot) для Топ-10 самых важных признаков.

*🤖 Помощь AI-тьютора:*
* `#XAI_TASK1_START` — для сортировки значений используйте `sort_values(ascending=False)`.


In [ ]:
# [MASTER SOLUTION]
if 'final_model' in locals() and hasattr(final_model, 'feature_importances_'):
    # 1. Извлечение важности признаков
    importances = final_model.feature_importances_
    features = X_test.columns
    
    # 2. Создание DataFrame для удобства сортировки
    fi_df = pd.DataFrame({'Feature': features, 'Importance': importances})
    fi_df = fi_df.sort_values(by='Importance', ascending=False).head(10)
    
    # 3. Визуализация Топ-10 факторов
    plt.figure(figsize=(10, 6))
    sns.barplot(data=fi_df, x='Importance', y='Feature', palette='viridis')
    plt.title("Топ-10 факторов, влияющих на целевое событие (Встроенный Feature Importance)", fontsize=14)
    plt.xlabel("Относительная важность признака (от 0 до 1)")
    plt.ylabel("Бизнес-метрика")
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Модель не загружена или не поддерживает атрибут feature_importances_.")


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 1.1. Достаньте атрибут final_model.feature_importances_
# TODO: 1.2. Создайте pd.DataFrame с признаками и их весами
# TODO: 1.3. Отсортируйте Топ-10 и постройте sns.barplot
raise NotImplementedError("Задание 1 не выполнено! Реализуйте построение Feature Importance.")

# Ваш код визуализации глобальной важности...


---

## 🛠 ЗАДАНИЕ 2: Глубокая аналитика векторов влияния (SHAP Summary Plot)
**Бизнес-контекст:** Базовый `Feature Importance` показывает *силу* влияния, но не показывает *направление* (например, высокий стаж клиента снижает риск оттока или повышает его?). Векторы Шепли (SHAP) решают эту проблему, отображая на графике распределение влияния каждого значения признака.

**Инструкция (TODO):**
1. Инициализируйте `shap.TreeExplainer` (или `Explainer`), передав в него `final_model`.
2. Рассчитайте SHAP-значения (`shap_values`) для тестовой выборки `X_test`.
3. Постройте график `shap.summary_plot()`.

*🤖 Помощь AI-тьютора:*
* `#XAI_TASK2_START` — расчет SHAP может занять некоторое время. Если данных много, для скорости можно передать `X_test.sample(1000)`.


In [ ]:
# [MASTER SOLUTION]
if 'final_model' in locals():
    print("⏳ Расчет векторов Шепли (SHAP Values)...")
    
    # Берем случайную подвыборку для ускорения расчета
    X_sample = X_test.sample(min(1000, X_test.shape[0]), random_state=42)
    
    # 1. Инициализация Explainer
    explainer = shap.TreeExplainer(final_model)
    
    # 2. Расчет SHAP-значений
    # Для бинарной классификации explainer может вернуть список массивов (по одному на класс)
    # или один массив. Нам нужен массив для класса 1 (Отток).
    shap_values = explainer.shap_values(X_sample)
    if isinstance(shap_values, list):
        shap_values_to_plot = shap_values[1]
    else:
        shap_values_to_plot = shap_values
        
    print("✅ Векторы рассчитаны. Генерация отчета...")
    
    # 3. Визуализация Summary Plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values_to_plot, X_sample, show=False)
    plt.title("Векторы влияния признаков (SHAP Summary Plot)", fontsize=15)
    plt.tight_layout()
    plt.show()


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 2.1. Инициализируйте explainer = shap.TreeExplainer(final_model)
# TODO: 2.2. Рассчитайте shap_values для X_test
# TODO: 2.3. Постройте shap.summary_plot()
raise NotImplementedError("Задание 2 не выполнено! Реализуйте глобальную SHAP-интерпретацию.")

# Ваш код расчета и визуализации SHAP...


---

## 🛠 ЗАДАНИЕ 3: Локальная интерпретация клиента (SHAP Waterfall / Force Plot)
**Бизнес-контекст:** Оператор колл-центра или менеджер по работе с клиентами не оперирует глобальными графиками. Ему нужно знать, почему конкретный клиент прямо сейчас попадает в зону риска, чтобы предложить ему персонализированную скидку. SHAP позволяет разобрать прогноз конкретного человека на слагаемые.

**Инструкция (TODO):**
1. Выберите одного "рискового" клиента из `X_test` (например, строку `X_test.iloc[0:1]`).
2. Извлеките базовое значение (expected value) из explainer.
3. Постройте `shap.plots.waterfall` или `shap.force_plot` для визуализации вклада признаков в конкретно этот прогноз.

*🤖 Помощь AI-тьютора:*
* `#XAI_TASK3_START` — для нового API SHAP проще всего использовать `shap.Explainer` (вместо TreeExplainer) и построить водопадный график `shap.plots.waterfall(shap_values_obj[0])`.


In [ ]:
# [MASTER SOLUTION]
if 'final_model' in locals():
    # Выбираем одного клиента (например, индекс 5)
    client_idx = 5
    client_data = X_test.iloc[[client_idx]]
    
    # Прогноз модели для этого клиента
    client_prob = final_model.predict_proba(client_data)[0][1]
    print(f"🔍 Анализ клиента по индексу {client_idx}. Риск оттока (вероятность): {client_prob:.2%}")
    
    # Используем общий Explainer, чтобы получить объект Explanation
    explainer_obj = shap.Explainer(final_model, X_train if 'X_train' in locals() else X_test)
    shap_values_obj = explainer_obj(client_data)
    
    # Если модель вернула многоклассовый объект, выбираем индекс класса 1
    if len(shap_values_obj.shape) > 2:
        shap_values_client = shap_values_obj[0, :, 1]
    else:
        shap_values_client = shap_values_obj[0]
    
    # Визуализация "водопада" факторов
    plt.figure(figsize=(10, 5))
    shap.plots.waterfall(shap_values_client, show=False)
    plt.title(f"Персонализированный разбор факторов риска клиента", fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 3.1. Выберите одного клиента из X_test
# TODO: 3.2. Рассчитайте объект shap_values для этой строки
# TODO: 3.3. Визуализируйте локальный профиль через shap.plots.waterfall